# Bài 10
Đây là notebook chứa mã nguồn đầy đủ của bài 10.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import pulp
import pyomo.environ as pyo
from scipy.optimize import linprog, minimize, milp, LinearConstraint, Bounds
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize as pymoo_minimize

from src.data_loader import get_data


In [ ]:
def solve_bai10(p_optimistic=0.30, p_baseline=0.45, p_pessimistic=0.20, first_stage_cap=65):
    total_p = p_optimistic + p_baseline + p_pessimistic or 1
    p_opt, p_base, p_pess = p_optimistic/total_p, p_baseline/total_p, p_pessimistic/total_p
    
    categories = ['I (Hạ tầng)', 'D (Số hóa)', 'AI', 'H (Nhân lực)']
    scenario_names = ['Lạc quan', 'Cơ sở', 'Bi quan']
    
    returns_s_arr = np.array([
        [1.10, 1.20, 1.80, 1.05],  # lạc quan
        [1.00, 1.10, 1.30, 0.95],  # cơ sở
        [0.80, 0.90, 0.60, 1.15],  # bi quan
    ])
    
    p_dict = {'s1': p_opt, 's2': p_base, 's3': p_pess}
    beta_base = {'I':1.00, 'D':1.17, 'AI':1.18, 'H':1.05}

    def build_sp_model():
        m = pyo.ConcreteModel()
        m.J = pyo.Set(initialize=['I','D','AI','H'])
        m.S = pyo.Set(initialize=['s1','s2','s3'])
        m.p = pyo.Param(m.S, initialize=p_dict)
        m.beta = pyo.Param(m.J, initialize=beta_base)
        
        beta_s_dict = {}
        for s_idx, s in enumerate(['s1', 's2', 's3']):
            beta_s_dict[(s, 'I')] = returns_s_arr[s_idx, 0]
            beta_s_dict[(s, 'D')] = returns_s_arr[s_idx, 1]
            beta_s_dict[(s, 'AI')] = returns_s_arr[s_idx, 2]
            beta_s_dict[(s, 'H')] = returns_s_arr[s_idx, 3]
            
        m.beta_s = pyo.Param(m.S, m.J, initialize=beta_s_dict)
        m.x = pyo.Var(m.J, within=pyo.NonNegativeReals, bounds=(5, 30))
        m.y = pyo.Var(m.S, m.J, within=pyo.NonNegativeReals)
        
        m.budget1 = pyo.Constraint(expr=sum(m.x[j] for j in m.J) <= first_stage_cap)
        def budget2_rule(m, s): return sum(m.y[s,j] for j in m.J) <= 15.0
        m.budget2 = pyo.Constraint(m.S, rule=budget2_rule)
        def ai_limit_rule(m, s): return m.y[s,'AI'] <= 0.5 * m.x['H']
        m.ai_limit = pyo.Constraint(m.S, rule=ai_limit_rule)
        def obj_rule(m):
            return sum(m.beta[j]*m.x[j] for j in m.J) + sum(m.p[s] * sum(m.beta_s[s,j]*m.y[s,j] for j in m.J) for s in m.S)
        m.obj = pyo.Objective(rule=obj_rule, sense=pyo.maximize)
        return m

    try:
        import pulp
        cbc_path = pulp.PULP_CBC_CMD().path
        solver = pyo.SolverFactory('cbc', executable=cbc_path)
    except Exception:
        solver = pyo.SolverFactory('cbc')

    try:
        # 1. Solve SP
        m_sp = build_sp_model()
        solver.solve(m_sp)
        sp_ok = True
        sp_value = pyo.value(m_sp.obj)
        x_sp = [pyo.value(m_sp.x['I']), pyo.value(m_sp.x['D']), pyo.value(m_sp.x['AI']), pyo.value(m_sp.x['H'])]
        
        # 2. Solve EV (Expected Value problem)
        m_ev = pyo.ConcreteModel()
        m_ev.J = pyo.Set(initialize=['I','D','AI','H'])
        m_ev.beta = pyo.Param(m_ev.J, initialize=beta_base)
        mean_beta_s = {j: sum(p_dict[s] * m_sp.beta_s[s,j] for s in ['s1','s2','s3']) for j in ['I','D','AI','H']}
        m_ev.mean_beta_s = pyo.Param(m_ev.J, initialize=mean_beta_s)
        m_ev.x = pyo.Var(m_ev.J, within=pyo.NonNegativeReals, bounds=(5, 30))
        m_ev.y = pyo.Var(m_ev.J, within=pyo.NonNegativeReals)
        m_ev.budget1 = pyo.Constraint(expr=sum(m_ev.x[j] for j in m_ev.J) <= first_stage_cap)
        m_ev.budget2 = pyo.Constraint(expr=sum(m_ev.y[j] for j in m_ev.J) <= 15.0)
        m_ev.ai_limit = pyo.Constraint(expr=m_ev.y['AI'] <= 0.5 * m_ev.x['H'])
        m_ev.obj = pyo.Objective(expr=sum(m_ev.beta[j]*m_ev.x[j] for j in m_ev.J) + sum(m_ev.mean_beta_s[j]*m_ev.y[j] for j in m_ev.J), sense=pyo.maximize)
        solver.solve(m_ev)
        ev_value = pyo.value(m_ev.obj)
        x_ev = [pyo.value(m_ev.x['I']), pyo.value(m_ev.x['D']), pyo.value(m_ev.x['AI']), pyo.value(m_ev.x['H'])]

        # Evaluate EV decision in SP environment (EEV)
        m_eev = build_sp_model()
        m_eev.x['I'].fix(x_ev[0])
        m_eev.x['D'].fix(x_ev[1])
        m_eev.x['AI'].fix(x_ev[2])
        m_eev.x['H'].fix(x_ev[3])
        solver.solve(m_eev)
        eev_value = pyo.value(m_eev.obj)

        # 3. Solve WS (Wait-and-See) for each scenario
        ws_values = []
        for s_idx, s in enumerate(['s1','s2','s3']):
            m_ws = pyo.ConcreteModel()
            m_ws.J = pyo.Set(initialize=['I','D','AI','H'])
            m_ws.x = pyo.Var(m_ws.J, within=pyo.NonNegativeReals, bounds=(5, 30))
            m_ws.y = pyo.Var(m_ws.J, within=pyo.NonNegativeReals)
            m_ws.budget1 = pyo.Constraint(expr=sum(m_ws.x[j] for j in m_ws.J) <= first_stage_cap)
            m_ws.budget2 = pyo.Constraint(expr=sum(m_ws.y[j] for j in m_ws.J) <= 15.0)
            m_ws.ai_limit = pyo.Constraint(expr=m_ws.y['AI'] <= 0.5 * m_ws.x['H'])
            beta_s_curr = {
                'I': returns_s_arr[s_idx, 0], 'D': returns_s_arr[s_idx, 1],
                'AI': returns_s_arr[s_idx, 2], 'H': returns_s_arr[s_idx, 3]
            }
            m_ws.obj = pyo.Objective(expr=sum(beta_base[j]*m_ws.x[j] for j in m_ws.J) + sum(beta_s_curr[j]*m_ws.y[j] for j in m_ws.J), sense=pyo.maximize)
            solver.solve(m_ws)
            ws_values.append(pyo.value(m_ws.obj))
        
        ws_expected = sum(p_dict[s]*ws_values[i] for i, s in enumerate(['s1','s2','s3']))

        # 4. Robust Optimization (Maximize Worst-Case Scenario)
        m_rob = pyo.ConcreteModel()
        m_rob.J = pyo.Set(initialize=['I','D','AI','H'])
        m_rob.S = pyo.Set(initialize=['s1','s2','s3'])
        m_rob.beta = pyo.Param(m_rob.J, initialize=beta_base)
        m_rob.x = pyo.Var(m_rob.J, within=pyo.NonNegativeReals, bounds=(5, 30))
        m_rob.y = pyo.Var(m_rob.S, m_rob.J, within=pyo.NonNegativeReals)
        m_rob.Z = pyo.Var() # Worst case profit
        
        m_rob.budget1 = pyo.Constraint(expr=sum(m_rob.x[j] for j in m_rob.J) <= first_stage_cap)
        def rob_budget2_rule(m, s): return sum(m.y[s,j] for j in m.J) <= 15.0
        m_rob.budget2 = pyo.Constraint(m_rob.S, rule=rob_budget2_rule)
        def rob_ai_limit_rule(m, s): return m.y[s,'AI'] <= 0.5 * m.x['H']
        m_rob.ai_limit = pyo.Constraint(m_rob.S, rule=rob_ai_limit_rule)
        
        def worst_case_rule(m, s):
            first = sum(m.beta[j]*m.x[j] for j in m.J)
            second = sum(m_sp.beta_s[s,j]*m.y[s,j] for j in m.J) # reuse m_sp.beta_s
            return m.Z <= first + second
        m_rob.worst_case = pyo.Constraint(m_rob.S, rule=worst_case_rule)
        m_rob.obj = pyo.Objective(expr=m_rob.Z, sense=pyo.maximize)
        
        solver.solve(m_rob)
        rob_value = pyo.value(m_rob.Z)
        x_rob = [pyo.value(m_rob.x['I']), pyo.value(m_rob.x['D']), pyo.value(m_rob.x['AI']), pyo.value(m_rob.x['H'])]

    except Exception as e:
        print("Bài 10 Error:", e)
        sp_ok = False
        sp_value = ev_value = eev_value = ws_expected = rob_value = 0.0
        x_sp = x_ev = x_rob = [0]*4

    vss = float(sp_value - eev_value)
    evpi = float(ws_expected - sp_value)

    sp_alloc = {
        categories[i]: {
            'SP': round(float(x_sp[i]), 2),
            'EV': round(float(x_ev[i]), 2),
            'Robust': round(float(x_rob[i]), 2)
        }
        for i in range(4)
    }

    return {
        'sp_value':    round(sp_value, 3),
        'ev_value':    round(ev_value, 3),
        'eev_value':   round(eev_value, 3),
        'ws_value':    round(ws_expected, 3),
        'rob_value':   round(rob_value, 3),
        'vss':         round(vss, 3),
        'evpi':        round(evpi, 3),
        'x_sp':        x_sp,
        'x_ev':        x_ev,
        'x_rob':       x_rob,
        'probabilities': {'optimistic': p_opt, 'baseline': p_base, 'pessimistic': p_pess},
        'sp_alloc':    sp_alloc,
        'feasible':    sp_ok,
    }

In [ ]:
if __name__ == '__main__':
    res = solve_bai10()
    # In ra một số key để kiểm tra
    if isinstance(res, dict):
        print(res.keys())